[← Back to 03 - Aggregation](<../03-aggregation.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 03 - Aggregation: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/03-aggregation-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TEMP TABLE lesson_orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    status TEXT NOT NULL,
    total INTEGER
);
INSERT INTO lesson_orders VALUES
    (1, 'Maria', 'completed', 120),
    (2, 'Maria', 'completed', 100),
    (3, 'Maria', 'cancelled', 90),
    (4, 'David', 'completed', 150),
    (5, 'David', 'completed', NULL),
    (6, 'Amina', 'completed', 200),
    (7, 'Amina', 'cancelled', 50);

7 rows inserted


## 1. At least $200

In [3]:
%%sql
SELECT
    customer_name,
    SUM(total) AS recorded_spend
FROM lesson_orders
WHERE status = 'completed'
GROUP BY
    customer_name
HAVING SUM(total) >= 200
ORDER BY
    customer_name;

customer_name,recorded_spend
Amina,200
Maria,220


Only the boundary changed: `>=` includes Amina, and the cancelled orders still aren't counted because the `WHERE` condition stayed the same, so Maria's cancelled $90 makes no difference. David's missing amount also remains unresolved, since changing a threshold doesn't repair the source data.

## 2. Cancelled orders per customer

In [4]:
%%sql
SELECT
    customer_name,
    COUNT(*) AS cancelled_orders,
    SUM(total) AS cancelled_value
FROM lesson_orders
WHERE status = 'cancelled'
GROUP BY
    customer_name
ORDER BY
    customer_name;

customer_name,cancelled_orders,cancelled_value
Amina,1,50
Maria,1,90


Amina and Maria, one cancelled order each. David doesn't appear at all, because `WHERE` removed all his rows before grouping, so no David group was ever formed. That's worth remembering: a grouped result only has rows for groups that still have at least one row after `WHERE`. If the report needs David listed with a zero, that takes a different technique, which `04 - JOINs` covers in its section on counting zeros.

## 3. Customers with a gap

In [5]:
%%sql
SELECT
    customer_name,
    COUNT(*) AS completed_orders,
    COUNT(total) AS known_totals
FROM lesson_orders
WHERE status = 'completed'
GROUP BY
    customer_name
HAVING COUNT(*) > COUNT(total)
ORDER BY
    customer_name;

customer_name,completed_orders,known_totals
David,2,1


Only David: two completed orders, one known total. `COUNT(*)` counts rows and `COUNT(total)` counts rows with a value, so when they differ, the group has at least one missing value. It's a useful data-quality check to run before trusting any `SUM` or `AVG`.

## 4. Stretch: customers with 25 or more orders (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [6]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [7]:
%%tsql
SELECT
    CustomerID,
    COUNT(*) AS orders
FROM Sales.SalesOrderHeader
GROUP BY
    CustomerID
HAVING COUNT(*) >= 25
ORDER BY
    orders DESC,
    CustomerID;

CustomerID,orders
11091,28
11176,28
11185,27
11200,27
11223,27
11262,27
11276,27
11277,27
11287,27
11300,27


Fourteen customers, led by two with 28 orders each. Notice the second sort column, `CustomerID`: several customers tie on their order count, and without a tie-breaker the order of tied rows isn't defined, so the list could come back in a different order next time. Whenever you sort on something that can tie, add a column that can't.

---

[← Back to 03 - Aggregation](<../03-aggregation.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).